In [1]:
import requests
import os
import json
import csv
from datetime import datetime

API_KEY = API_KEY = "eyJpdiI6IkU0aTRhaGN5SEEzd1p1WDlMRTBNWkE9PSIsInZhbHVlIjoidGhMaG1rWWs0cmlxd3MwOFNEcVFiRkowVTBoWTdaOElWZ2djaUlHY25kb2Z5MnRaMTdhZm4xMnZJbFBmTlZRbkcvd3dGWTFQaDJ1MDhpMG5URkd1TUE9PSIsIm1hYyI6IjA2OGQxZjIxNThmZTg0MWE1NmIzOWQwODdlM2RhZTM3M2E1NGE1ZTc1ZWVkYmJmMDRhNTFkNGJiMmNkNGE5NDUiLCJ0YWciOiIifQ=="

BASE_URL = "https://tools.flipperforce.com/api/v1"

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Accept": "application/json",
    "User-Agent": "python-requests/flipperforce-pull"
}

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# CSV output directory for Amplify frontend public data
csv_dir = os.path.abspath(
    os.path.join(
        "cpc-aws-v2",
        "public",
        "data"
    )
)

os.makedirs(csv_dir, exist_ok=True)

# Raw JSON archive disabled for now
# base_dir = os.path.join("archive", "flipperforce", timestamp)
# raw_dir = os.path.join(base_dir, "raw")
# os.makedirs(raw_dir, exist_ok=True)


# def save_json(data, filename):
#     path = os.path.join(raw_dir, filename)
#     with open(path, "w", encoding="utf-8") as f:
#         json.dump(data, f, indent=2)
#     print(f"Saved JSON: {path}")


def write_csv(rows, filename, fieldnames=None):
    path = os.path.join(csv_dir, filename)

    if not rows:
        print(f"Skipped CSV (no rows): {path}")
        return

    if fieldnames is None:
        keys = set()
        for row in rows:
            keys.update(row.keys())
        fieldnames = list(keys)

    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)

    print(f"Saved CSV : {path}")


def request_json(endpoint, params=None):
    url = f"{BASE_URL}{endpoint}"
    print("=" * 100)
    print(f"GET {url}")

    r = requests.get(url, headers=HEADERS, params=params, timeout=60)
    print(f"Status: {r.status_code}")

    try:
        payload = r.json()
    except Exception:
        payload = {"raw_text": r.text}

    if r.status_code >= 400:
        preview = json.dumps(payload, indent=2)[:2000]
        print(preview)
        return r.status_code, payload

    return r.status_code, payload


def extract_projects(payload):
    if isinstance(payload, dict):
        data = payload.get("data")
        if isinstance(data, list):
            return data
    return []


def flatten_project(project):
    return {
        "project_uuid": project.get("uuid"),
        "workspace_uuid": project.get("workspace_uuid"),
        "created_at": project.get("created_at"),
        "updated_at": project.get("updated_at"),
        "name": project.get("name"),
        "full_address": project.get("full_address"),
        "address_1": project.get("address_1"),
        "address_2": project.get("address_2"),
        "city": project.get("city"),
        "state": project.get("state"),
        "postal_code": project.get("postal_code"),
        "country": project.get("country"),
        "lat": project.get("lat"),
        "lng": project.get("lng"),
        "investment_strategy": project.get("investment_strategy"),
        "stage": project.get("stage"),
        "type": project.get("type"),
        "style": project.get("style"),
        "square_feet": project.get("square_feet"),
        "beds": project.get("beds"),
        "baths": project.get("baths"),
        "year_built": project.get("year_built"),
        "featured_image_url": project.get("featured_image_url"),
        "permissions_json": json.dumps(project.get("permissions", []))
    }


def flatten_photo_rows(project_uuid, payload):
    rows = []

    data = payload.get("data", {})
    photos_by_date = data.get("photos_by_date", [])

    for group in photos_by_date:
        group_date = group.get("group_date")
        photos = group.get("photos", [])

        for photo_item in photos:
            photo_obj = photo_item.get("photo", {}) or {}
            source_url = photo_obj.get("source_url", {}) or {}
            preview_url = photo_obj.get("preview_url", {}) or {}

            rows.append({
                "project_uuid": project_uuid,
                "group_date": group_date,
                "photo_log_uuid": photo_item.get("uuid"),
                "photo_order": photo_item.get("order"),
                "created_at": photo_item.get("created_at"),
                "photo_date": photo_item.get("photo_date"),
                "photo_description": photo_item.get("photo_description"),
                "photo_uuid": photo_obj.get("uuid"),
                "photo_name": photo_obj.get("name"),
                "photo_size": photo_obj.get("size"),
                "photo_type": photo_obj.get("type"),
                "source_view_url": source_url.get("view"),
                "source_download_url": source_url.get("download"),
                "preview_thumbnail_url": preview_url.get("thumbnail"),
                "preview_large_url": preview_url.get("large")
            })

    return rows


def flatten_receipt_rows(project_uuid, payload):
    rows = []

    data = payload.get("data", [])

    if isinstance(data, list):
        receipts = data
    elif isinstance(data, dict):
        receipts = data.get("receipts", [])
    else:
        receipts = []

    for receipt in receipts:
        document = receipt.get("document", {}) or {}
        file_info = receipt.get("file", {}) or {}

        rows.append({
            "project_uuid": project_uuid,
            "receipt_uuid": receipt.get("uuid"),
            "created_at": receipt.get("created_at"),
            "updated_at": receipt.get("updated_at"),
            "receipt_date": receipt.get("receipt_date") or receipt.get("date"),
            "vendor": receipt.get("vendor") or receipt.get("vendor_name"),
            "amount": receipt.get("amount") or receipt.get("total"),
            "currency": receipt.get("currency", "USD"),
            "category": receipt.get("category"),
            "description": receipt.get("description") or receipt.get("notes"),
            "expense_uuid": receipt.get("expense_uuid"),
            "document_uuid": document.get("uuid") or file_info.get("uuid"),
            "document_name": document.get("name") or file_info.get("name"),
            "document_url": document.get("url") or document.get("source_url") or file_info.get("url"),
            "status": receipt.get("status"),
            "tags": json.dumps(receipt.get("tags", [])),
        })

    return rows


def main():
    print("Starting Flipper Force CSV pull...")
    print(f"CSV output folder: {csv_dir}")

    status, account_payload = request_json("/user/account")
    if status != 200:
        print("Auth failed. Exiting.")
        return

    # save_json(account_payload, "user_account.json")

    status, projects_payload = request_json("/project/list")
    if status != 200:
        print("Project list pull failed. Exiting.")
        return

    # save_json(projects_payload, "projects_list.json")

    projects = extract_projects(projects_payload)
    print(f"Projects found: {len(projects)}")

    project_rows = [flatten_project(p) for p in projects]

    write_csv(project_rows, "projects_v2.csv", fieldnames=[
        "project_uuid",
        "workspace_uuid",
        "created_at",
        "updated_at",
        "name",
        "full_address",
        "address_1",
        "address_2",
        "city",
        "state",
        "postal_code",
        "country",
        "lat",
        "lng",
        "investment_strategy",
        "stage",
        "type",
        "style",
        "square_feet",
        "beds",
        "baths",
        "year_built",
        "featured_image_url",
        "permissions_json"
    ])

    all_photo_rows = []
    all_receipt_rows = []
    manifest_rows = []

    for i, project in enumerate(projects, start=1):
        project_uuid = project.get("uuid")
        project_name = project.get("name")

        if not project_uuid:
            continue

        print("-" * 100)
        print(f"[{i}/{len(projects)}] Processing {project_name} | {project_uuid}")

        print("  → Pulling photo log...")
        status_photo, photo_payload = request_json(f"/project/{project_uuid}/photo-log")

        print("  → Pulling receipts...")
        status_receipt, receipt_payload = request_json(f"/project/{project_uuid}/receipts/list")

        manifest_rows.append({
            "project_uuid": project_uuid,
            "project_name": project_name,
            "photo_log_status": status_photo,
            "receipts_status": status_receipt
        })

        if status_photo == 200:
            # save_json(photo_payload, f"project_{project_uuid}_photo_log.json")
            all_photo_rows.extend(flatten_photo_rows(project_uuid, photo_payload))

        if status_receipt == 200:
            # save_json(receipt_payload, f"project_{project_uuid}_receipts.json")
            all_receipt_rows.extend(flatten_receipt_rows(project_uuid, receipt_payload))

    write_csv(all_photo_rows, "project_photo_log_v2.csv", fieldnames=[
        "project_uuid",
        "group_date",
        "photo_log_uuid",
        "photo_order",
        "created_at",
        "photo_date",
        "photo_description",
        "photo_uuid",
        "photo_name",
        "photo_size",
        "photo_type",
        "source_view_url",
        "source_download_url",
        "preview_thumbnail_url",
        "preview_large_url"
    ])

    write_csv(all_receipt_rows, "project_receipts.csv", fieldnames=[
        "project_uuid",
        "receipt_uuid",
        "created_at",
        "updated_at",
        "receipt_date",
        "vendor",
        "amount",
        "currency",
        "category",
        "description",
        "expense_uuid",
        "document_uuid",
        "document_name",
        "document_url",
        "status",
        "tags"
    ])

    write_csv(manifest_rows, "pull_manifest_v2.csv", fieldnames=[
        "project_uuid",
        "project_name",
        "photo_log_status",
        "receipts_status"
    ])

    print("=" * 100)
    print("DONE")
    print(f"CSV dir        : {csv_dir}")
    print(f"Projects pulled: {len(project_rows)}")
    print(f"Photos pulled  : {len(all_photo_rows)}")
    print(f"Receipts pulled: {len(all_receipt_rows)}")


if __name__ == "__main__":
    main()

Starting Flipper Force CSV pull...
CSV output folder: C:\CPC\cpc-aws-v2\cpc-aws-v2\public\data\cpc-aws-v2\public\data
GET https://tools.flipperforce.com/api/v1/user/account
Status: 200
GET https://tools.flipperforce.com/api/v1/project/list
Status: 200
Projects found: 62
Saved CSV : C:\CPC\cpc-aws-v2\cpc-aws-v2\public\data\cpc-aws-v2\public\data\projects_v2.csv
----------------------------------------------------------------------------------------------------
[1/62] Processing 2560 Sayers Road | 6a2cd601-d352-47e6-9f7b-2895680ede08
  → Pulling photo log...
GET https://tools.flipperforce.com/api/v1/project/6a2cd601-d352-47e6-9f7b-2895680ede08/photo-log
Status: 200
  → Pulling receipts...
GET https://tools.flipperforce.com/api/v1/project/6a2cd601-d352-47e6-9f7b-2895680ede08/receipts/list
Status: 200
----------------------------------------------------------------------------------------------------
[2/62] Processing 5278 Willow Ridge Lane | fc03e317-7886-41fe-8df6-ddaab2feaa7f
  → Pullin